<a href="https://colab.research.google.com/github/samyyy2311/Unsupervised_Learning/blob/main/Unsupervised_Learning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Assignment - Customer Segmentation using Unsupervised Learning

**Dataset:** Online Retail II - [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/502/online+retail+ii)

**Source:** Chen, D. (2012). Online Retail II [Dataset]. UCI Machine Learning Repository.

**Author:** Samarth Lad - 2025EB1100248

# Task A: Problem Framing and Data Understanding

## Step 1: Loading the Dataset

**Why:** Before I do anything else, I need to get the raw data into Python and see its basic size and structure. I'm downloading it straight from the UCI page so the notebook can run from start to finish without needing anyone to manually upload a file. The original file stays untouched in the raw folder.

**What the code does:** Sets up the libraries and a random seed, downloads and unzips the data from UCI, reads both sheets (2009-2010 and 2010-2011), joins them into one table called df, and shows the first 5 rows.

In [38]:
import pandas as pd
import numpy as np
import urllib.request, zipfile, io

# setting a seed so results don't change every time I rerun the notebook
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# Downloading the raw data from uci, the original file is saved untouched in the raw folder
url = 'https://archive.ics.uci.edu/static/public/502/online+retail+ii.zip'
zipfile.ZipFile(io.BytesIO(urllib.request.urlopen(url).read())).extractall('raw')

# Reading both sheets (2009-2010 and 2010-2011) and joining them into one table
sheets = pd.read_excel('raw/online_retail_II.xlsx', sheet_name=None)
df = pd.concat(sheets.values(), ignore_index=True)
df.head()

### What I found

- The two sheets together have **1,067,371 rows** and **8 columns** (525,461 + 541,910)
- The columns are `Invoice`, `StockCode`, `Description`, `Quantity`, `InvoiceDate`, `Price`, `Customer ID` and `Country`
- Each row is one product line on an invoice, not one customer, so I will have to group rows into customers before clustering

## Step 2a: Checking Data Types and Date Range

**Why:** Before I clean anything, I want to know what type each column is stored as and what time period the data covers. Wrong types (like dates stored as text) would cause problems later when I build features like recency.

**What the code does:** Shows each column name, how many non-null values it has and its data type, then shows the first and last invoice date.

In [ ]:
df.info()
df['InvoiceDate'].agg(['min', 'max'])

### What I found

- `InvoiceDate` was read as a proper date, so I don't need to convert it
- The data runs from **1 December 2009 to 9 December 2011**, about two years of orders
- `Invoice` is stored as text, because cancellation invoices start with the letter C
- `Customer ID` is stored as a decimal number, which usually means some values are missing, so I'll check that next

## Step 2b: Checking Missing Values

**Why:** Missing values can't be used in distance calculations. Customer ID matters most here, because it decides which rows I can group into customers.

**What the code does:** Counts how many values are missing in each column.

In [ ]:
df.isna().sum()

### What I found

- **243,007 rows (22.8%) have no Customer ID**, so I can't link them to any customer. This is the biggest data problem for building a customer-level table
- `Description` is missing in 4,382 rows (0.4%), which doesn't matter much since I won't cluster on product text
- No other column has missing values
- I will decide what to do with the rows without a Customer ID in Task B

## Step 2c: Checking Duplicate Rows

**Why:** Duplicate lines would count the same purchase twice and make a customer look like they spent more than they did.

**What the code does:** Counts the rows that are exact copies of another row.

In [ ]:
df.duplicated().sum()

### What I found

- There are **34,335 fully duplicated rows (3.2%)**
- Some of these may be real repeat scans of the same item on one invoice, so I'll look at them more closely before deciding to remove them

## Step 2d: Counting Customers, Invoices, Products and Countries

**Why:** The assignment wants one row per customer, so I need to know how many customers there are compared to invoices and products. I also want to see how the countries are spread, in case one country dominates.

**What the code does:** Counts the unique values in four columns, then shows the five countries with the most rows.

In [ ]:
print(df[['Customer ID', 'Invoice', 'StockCode', 'Country']].nunique())
print()
print(df['Country'].value_counts().head(5))

### What I found

- There are **5,942 unique customers**, 53,628 invoices, 5,305 products and 43 countries
- That is about 139 rows per customer on average (824,364 rows with an ID divided by 5,942 customers), so summarising each customer into a few features will be a big reduction
- The United Kingdom has 981,330 rows, which is **91.9%** of the whole dataset. The next largest are EIRE (17,866), Germany (17,624), France (14,330) and the Netherlands (5,140)
- Because the data is mostly one country, I'll be careful about using `Country` as a clustering feature

## Step 2e: Checking Cancellations and Zero or Negative Values

**Why:** Cancellations and refunds are not purchases, and the assignment asks me to explain how I treat them. Zero or negative prices and quantities also need a closer look.

**What the code does:** Prints three counts in order: invoices that start with C (cancellations), rows with a Quantity of 0 or less, and rows with a Price of 0 or less.

In [ ]:
# Cancellations start with the letter C
inv = df['Invoice'].astype(str)
print('Cancellations:', inv.str.startswith('C').sum())
print('Quantity <= 0:', (df['Quantity'] <= 0).sum())
print('Price <= 0:', (df['Price'] <= 0).sum())

### What I found

- **19,494 rows (1.8%)** are cancellations
- **22,950 rows (2.2%)** have a Quantity of 0 or less, which is more than the number of cancellations, so some negative quantities are not cancellations. I'll check what they are in Step 3b
- **6,207 rows (0.6%)** have a Price of 0 or less

In [ ]:
df[['Quantity', 'Price']].describe(percentiles=[.01, .99]).round(2)

### What I found

- `Quantity` has a median of 3 and a mean of about 9.9, but the maximum is 80,995 and the minimum is -80,995. The 1st and 99th percentiles are only -3 and 100
- `Price` has a median of 2.10 and a mean of about 4.65, but the maximum is 38,970 and the minimum is -53,594.36, which is clearly not a normal sale
- Both columns are very skewed, with a few extreme rows far from everything else. I'll need to handle these later, for example with a log transform or robust scaling

## Step 3a: Odd Invoice Numbers

**Why:** Normal invoice numbers are digits only, or digits with a C in front for cancellations. Anything else could be an adjustment or a non-sale entry, so I want to see if any invoices don't follow that pattern.

**What the code does:** Shows the rows whose invoice number doesn't match either pattern.

In [ ]:
# Showing the invoices that don't follow the normal pattern
df[~inv.str.match(r'^C?\d+$')]

### What I found

- Six invoices don't follow the normal pattern. All of them have stock code `B`, the description "Adjust bad debt", a Quantity of 1, no Customer ID, and are from the United Kingdom
- These are accounting entries, not sales. Their prices are -53,594.36, -44,031.79, -38,925.87, 11,062.06 and -11,062.06 (twice)
- The extreme minimum Price I saw in Step 2f comes from these rows. I will remove them

## Step 3b: Negative Quantity That Isn't a Cancellation

**Why:** There are more rows with a Quantity of 0 or less (22,950) than cancellations (19,494), so some of them must be something else.

**What the code does:** Counts the negative-quantity rows that are not cancellations, then how many of those have a Price of 0 and how many have no Customer ID.

In [ ]:
# Negative quantity rows that are not cancellations
neg_not_c = df[(df['Quantity'] <= 0) & (~inv.str.startswith('C'))]
print('Rows:', len(neg_not_c))
print('With price 0:', neg_not_c['Price'].eq(0).sum())
print('With no customer id:', neg_not_c['Customer ID'].isna().sum())

### What I found

- Six invoices don't follow the normal pattern. All of them have stock code `B`, the description "Adjust bad debt", a Quantity of 1, no Customer ID, and are from the United Kingdom
- These are accounting entries, not sales. Their prices are -53,594.36, -44,031.79, -38,925.87, 11,062.06 and -11,062.06 (twice)
- The extreme minimum Price I saw in Step 2f comes from these rows. I will remove them

## Step 3c: Zero Price Rows

**Why:** A price of 0 isn't a normal sale, so I want to know how many there are and whether they belong to real customers.

**What the code does:** Counts the zero-price rows, then how many of them have no Customer ID.

In [ ]:
# Rows where the price is zero
zero_price = df[df['Price'] == 0]
print('Zero price rows:', len(zero_price))
print('With no customer id:', zero_price['Customer ID'].isna().sum())

### What I found

- **6,202 rows** have a Price of exactly 0, and 6,131 of them have no Customer ID. The other 71 do have one
- Adding the 5 negative prices from Step 3a gives 6,207, which is exactly the number of rows with a Price of 0 or less in Step 2e
- A zero-price row has no spending to measure, so I plan to remove all of them

## Step 3d: Stock Codes That Aren't Products

**Why:** Some stock codes with no digits are usually fees, postage or adjustments, not things a customer actually bought. Keeping them would give a wrong picture of buying behaviour.

**What the code does:** Lists every stock code that contains no digits, with how many rows each one has.

In [ ]:
# Stock codes with no digits are usually fees or postage, not real products
sc = df['StockCode'].astype(str)
df[~sc.str.contains(r'\d')]['StockCode'].value_counts()

### What I found

- 17 stock codes have no digits, covering **5,579 rows (0.5%)**. The biggest are `POST` (2,122), `DOT` (1,446) and `M` (1,421)
- Judging by their names, `POST`, `DOT`, `M`, `D`, `S`, `BANK CHARGES`, `ADJUST`, `AMAZONFEE`, `CRUK` and `B` look like postage, fees, discounts or manual adjustments, not products
- A few, like `DCGSSGIRL`, `DCGSSBOY`, `DCGSLGIRL`, `DCGSLBOY`, `PADS` and `GIFT`, might be real products, so I'll check their descriptions before removing anything

## Step 3e: Duplicates Among Known Customers

**Why:** The 34,335 duplicates include rows with no Customer ID, which I'm likely to leave out anyway. I want to know how many duplicates are left among rows that do have an ID.

**What the code does:** Counts duplicate rows after keeping only the rows that have a Customer ID.

In [ ]:
# Duplicates among rows that do have a customer id
df[df['Customer ID'].notna()].duplicated().sum()

### What I found

- **26,479 duplicate rows** remain among rows that have a Customer ID, which is 3.2% of those 824,364 rows. The other 7,856 duplicates had no ID
- These matter because they would inflate a real customer's spending and order counts
- An exact copy matches on invoice, product, quantity, time, price and customer, so I plan to remove them in Task B. A few could be genuine repeat scans, which I'll mention as a limitation

## Step 3f: Checking the Unclear Stock Codes

**Why:** In Step 3d, some codes could be real products or fees. I don't want to remove real sales by mistake.

**What the code does:** Shows the description, median price and number of rows for each unclear code.

In [ ]:
# Checking what the unclear stock codes actually are
unclear = ['DCGSSGIRL', 'DCGSSBOY', 'DCGSLGIRL', 'DCGSLBOY', 'PADS', 'GIFT', 'CRUK']
df[df['StockCode'].isin(unclear)].groupby('StockCode').agg(
    description=('Description', 'first'),
    median_price=('Price', 'median'),
    rows=('Price', 'size'))

### What I found

- `CRUK` is "CRUK Commission" with a median value of 471.77, so it is a fee and not a product
- `DCGSLBOY`, `DCGSLGIRL` and `GIFT` have one row each, no description and a price of 0, so the zero-price rule will remove them anyway
- `DCGSSBOY` and `DCGSSGIRL` (48 rows) show the description "update", which is not a product name. I'm not sure what they are, but with so few rows it makes no real difference, so I'll treat them as non-products
- `PADS` is a real product ("PADS TO MATCH ALL CUSHIONS") with a tiny price of 0.001 and only 19 rows, so I'll keep it
- Final list of non-product codes to remove: `POST`, `DOT`, `M`, `m`, `D`, `S`, `BANK CHARGES`, `ADJUST`, `AMAZONFEE`, `CRUK`, `B`, `DCGSSBOY`, `DCGSSGIRL`

## Step 4: Defining the Problem and the Analysis Unit

**Practical problem:** According to the UCI description, this is a UK-based online retailer that sells gift-ware, and many of its customers are wholesalers. A shop like this can't treat 5,942 customers the same way. I want to group customers by how they buy, so the business can see what types of customers it actually has.

**Analysis unit:** One row is one customer (`Customer ID`), built by summarising all of that customer's invoice lines across the two years. The raw data has one row per product line, so each customer's lines have to be aggregated first. This is one table, so there are no joins. I'll show the aggregation and check that each final row maps to exactly one customer in Task B.

**Intended user:** The marketing or customer team.

**Decision it could help with:** Deciding which customers get a retention offer, which get a win-back message, and which get wholesale-style account handling.

**Why clustering fits:** The data has no existing customer segment and no outcome I'm trying to predict, so this is not a classification problem. I want the groups to come from behaviour itself. The only status field is the C prefix on cancellation invoices, which describes a transaction and not a customer segment. I'll use it to handle returns and not as an answer for the algorithms to find. If I used an existing label as the target, I would just be checking my own labels, not discovering anything.

## Step 5: Data-Quality Summary and Column Types

**Data-quality summary (from Steps 2 and 3):**

| Issue | Count | Planned action |
|---|---|---|
| Rows / columns | 1,067,371 / 8 | None |
| No Customer ID | 243,007 (22.8%) | Remove, can't be assigned to a customer |
| Missing Description | 4,382 (0.4%) | Ignore, not used for clustering |
| Exact duplicate rows | 34,335 (26,479 with a Customer ID) | Remove |
| Cancellation rows | 19,494 (1.8%) | Handle separately from purchases |
| Negative quantity, not a cancellation | 3,457 | Remove, all have price 0 and no ID |
| Price of exactly 0 | 6,202 | Remove |
| "Adjust bad debt" invoices | 6 | Remove, accounting entries |
| Non-product stock codes | 13 codes | Remove |
| Very skewed Quantity and Price | max 80,995 and 38,970 | Transform before scaling |

**Column types:**

| Column | Type | Treatment |
|---|---|---|
| `Invoice` | Identifier | Not a feature, C prefix marks cancellations |
| `StockCode` | Identifier | Not a feature, used only to remove non-products |
| `Description` | Text | Not used |
| `Quantity` | Discrete numeric | Used to build features |
| `InvoiceDate` | Date/time | Used to build recency and frequency |
| `Price` | Continuous numeric | Unit price in pounds, used to build spend |
| `Customer ID` | Identifier | The key for grouping rows into customers |
| `Country` | Nominal | Kept for describing clusters afterwards |

**Cases where the right treatment is not obvious:**

- `Customer ID` looks like a number but is an identifier, so averaging or scaling it would be meaningless. It is stored as a decimal only because of the missing values
- `Invoice` looks numeric but is text, and the C prefix carries meaning, so I can't just convert it to a number
- `Quantity` is a count, but it goes negative for returns and means different things for different products (one bulk item versus one big piece)
- `Price` is per unit, not per line, so the real spend has to be calculated as Quantity times Price
- `Country` is 91.9% United Kingdom, so one-hot encoding it would add many columns that are almost all zeros

## Step 6: Columns Kept Out of Clustering

**Why:** The assignment asks me to decide which fields should not go into the algorithms, because every column I include changes what "similar" means.

**Excluded from the clustering input:**

- `Customer ID`, `Invoice` and `StockCode` are identifiers, so they only help to group rows or build features
- `Description` is free text, so I won't use it
- Raw `Quantity`, `Price` and `InvoiceDate` rows are not used directly, only through customer-level features such as recency, frequency and spend

**Kept for describing the clusters afterwards only:**

- `Country`, so I can see whether clusters differ by country without it deciding the groups

This keeps the measure of similarity based on buying behaviour only.

# Task B: Data Preprocessing and Exploratory Analysis

## Step 7: Making a Working Copy

**Why:** The assignment says to keep the raw data untouched, so I'm doing all the cleaning on a copy called clean. df stays exactly as it was downloaded.

**What the code does:** Copies df into clean and prints how many rows it has.

In [ ]:
clean = df.copy()
print('Rows in working copy:', len(clean))

## Step 8: Removing Rows With No Customer ID

**Why:** The goal is to group customers, and a row with no Customer ID can't be linked to anyone. Keeping these rows would not tell me anything about a customer, so I'm removing them. After this there are no missing IDs, so I'm also changing Customer ID from a decimal to a whole number.

**What the code does:** Keeps only rows that have a Customer ID, converts the ID to an integer, and prints how many rows were removed and how many are left.

In [ ]:
# Rows without a customer id can't be assigned to any customer
before = len(clean)
clean = clean[clean['Customer ID'].notna()].copy()
clean['Customer ID'] = clean['Customer ID'].astype(int)
print('Removed:', before - len(clean), '| Rows left:', len(clean))

### What I found

- **243,007 rows (22.8%)** were removed because they had no Customer ID, leaving 824,364 rows
- `Customer ID` is now a whole number

## Step 9: Checking for Inconsistent Labels

**Why:** If the same category is written two ways, like a country spelled differently or a stock code in upper and lower case, the data would treat them as two separate things.

**What the code does:** Lists every country name, then prints the number of unique stock codes as written and the number after making them all upper case.

In [ ]:
# Checking stock code spelling and country names
print(clean['StockCode'].nunique(), clean['StockCode'].astype(str).str.upper().nunique())
clean['Country'].value_counts()

### What I found

- The number of unique stock codes is the same before and after making them upper case, so there are no case spelling problems
- Each country appears under one name, so there are no spelling variants of the same country
- A few labels are not normal country names: `Unspecified`, `European Community`, `RSA` (South Africa) and `EIRE` (Ireland). I'll leave them as they are, because I'm not using `Country` as a clustering feature

## Step 10: Removing Exact Duplicate Rows

**Why:** In Step 3e I found duplicate rows among customers who have an ID. A row that is an exact copy of another one (same invoice, product, quantity, time, price and customer) would count the same purchase twice and inflate that customer's spending.

**What the code does:** Removes the repeated copies, keeping the first one, and prints how many rows were removed and how many are left.

In [ ]:
# Keeping the first copy of every exact duplicate
before = len(clean)
clean = clean.drop_duplicates()
print('Removed:', before - len(clean), '| Rows left:', len(clean))

### What I found

- **26,479 duplicate rows** were removed, leaving 797,885 rows
- This matches the count I found in Step 3e

## Step 11: Removing Non-Product Stock Codes

**Why:** Stock codes like postage, bank charges and manual adjustments are not things a customer bought. Keeping them would make customers look like they spend or order differently from how they really shop. I picked this list from Steps 3d and 3f.

**What the code does:** Removes the rows whose stock code is in the non-product list, and prints how many rows were removed and how many are left.

In [ ]:
# Stock codes that are fees, postage or adjustments, not real products
non_products = ['POST', 'DOT', 'M', 'm', 'D', 'S', 'BANK CHARGES', 'ADJUST',
                'AMAZONFEE', 'CRUK', 'B', 'DCGSSBOY', 'DCGSSGIRL']
before = len(clean)
clean = clean[~clean['StockCode'].isin(non_products)]
print('Removed:', before - len(clean), '| Rows left:', len(clean))

### What I found

- **3,368 rows** with non-product stock codes were removed, leaving 794,517 rows

## Step 12: Removing Zero Price Rows

**Why:** A row with a price of 0 has no spending to measure, so it would only add items to a customer's basket without adding any value. Most of these rows had no Customer ID and are already gone, so this removes what is left.

**What the code does:** Keeps only rows with a Price above 0, and prints how many rows were removed and how many are left.

In [ ]:
# A price of zero is not a real sale
before = len(clean)
clean = clean[clean['Price'] > 0]
print('Removed:', before - len(clean), '| Rows left:', len(clean))

### What I found

- Only **63 zero-price rows** were left to remove, because most were already gone in the earlier steps. This leaves 794,454 rows

## Step 13: Separating Sales From Returns

**Why:** A cancellation is not a purchase, so mixing it into the sales would reduce a customer's spend and quantity in a confusing way. I'm splitting the data into sales and returns. Sales will be used for spending and order features, and returns will only be used for a return-related feature later.

**What the code does:** Splits clean into sales and returns using the C at the start of the invoice number, then prints the rows in each, two checks that nothing is in the wrong table, and the number of customers.

In [ ]:
# Cancellation invoices start with the letter C
is_cancel = clean['Invoice'].astype(str).str.startswith('C')
returns = clean[is_cancel]
sales = clean[~is_cancel]
print('Sales rows:', len(sales), '| Return rows:', len(returns))
print('Sales with quantity <= 0:', (sales['Quantity'] <= 0).sum())
print('Returns with quantity > 0:', (returns['Quantity'] > 0).sum())
print('Customers in sales:', sales['Customer ID'].nunique())
print('Customers in clean data:', clean['Customer ID'].nunique())

### What I found

- The data split into **776,857 sales rows** and **17,597 return rows**, which adds up to the 794,454 rows left after cleaning
- Both checks came out as 0, so there are no sales with a Quantity of 0 or less and no returns with a positive Quantity. The split is clean
- 5,876 of the original 5,942 customers are still in the data. 66 customers lost all their rows during cleaning
- 5,853 customers have at least one sale. The other **23 customers only have returns** and no purchases, so I can't measure their buying behaviour. I will leave them out of the customer table, which makes the final table 5,853 customers

## Step 14: Setting the Snapshot Date and the Value of Each Line

**Why:** To measure recency I need a "today" to count back from, and to measure spending I need the value of each line. The data ends on 9 December 2011, so I'm using the day after the last invoice as the snapshot date. The price is per unit, so the value of a line is Quantity times Price.

**What the code does:** Sets the snapshot date, then adds a line_value column (in pounds) to the sales and returns tables.

In [ ]:
# One day after the last invoice, used to measure how recently a customer bought
snapshot = clean['InvoiceDate'].max() + pd.Timedelta(days=1)
print('Snapshot date:', snapshot)

# Value of each line is quantity times unit price
sales = sales.copy()
sales['line_value'] = sales['Quantity'] * sales['Price']
returns = returns.copy()
returns['line_value'] = returns['Quantity'].abs() * returns['Price']

### What I found

- The last invoice is on 9 December 2011, so the snapshot date is 10 December 2011. Recency is counted back from this date

## Step 15: Building One Row Per Customer

**Why:** The assignment says one row should be one customer, and the data currently has one row per product line. I'm summarising each customer's purchases into eight features that describe how they buy: how recently, how often, how much, how big their orders are, how varied their products are, how long they've been a customer, and how much they return.

**What the code does:** Groups the sales table by Customer ID and calculates the features, adds the return ratio from the returns table, and keeps the eight final columns in a table called customers.

In [ ]:
# Summarising each customer's sales into one row
features = sales.groupby('Customer ID').agg(
    last_purchase=('InvoiceDate', 'max'),
    first_purchase=('InvoiceDate', 'min'),
    frequency=('Invoice', 'nunique'),
    total_spend=('line_value', 'sum'),
    total_units=('Quantity', 'sum'),
    unique_products=('StockCode', 'nunique'))

features['recency_days'] = (snapshot - features['last_purchase']).dt.days
features['customer_age_days'] = (snapshot - features['first_purchase']).dt.days
features['avg_order_value'] = features['total_spend'] / features['frequency']
features['avg_basket_units'] = features['total_units'] / features['frequency']

# Returned value as a share of spend, 0 for customers with no returns
returned_value = returns.groupby('Customer ID')['line_value'].sum()
features['return_ratio'] = (returned_value / features['total_spend']).reindex(features.index).fillna(0)

cols = ['recency_days', 'frequency', 'total_spend', 'avg_order_value',
        'avg_basket_units', 'unique_products', 'customer_age_days', 'return_ratio']
customers = features[cols]
print(customers.shape)
customers.head()

### What I found

- The customer table has **5,853 rows and 8 columns**, one row per customer
- Customers look very different from each other. Customer 12347 bought 2 days before the snapshot, in 8 orders, with 126 different products. Customer 12348 bought 75 days before, in 5 orders, with 24 products
- Customer 12346 is unusual. Their total spend is 77,556.46, but their return ratio is roughly 0.99, so almost everything they bought was returned. A profile like this could look like a top spender when it isn't, so I'll watch for it when I look at outliers

## Step 16: Checking the Customer Table

**Why:** The assignment asks me to check that each final row is exactly one customer and that building the table didn't create duplicates or gaps.

**What the code does:** Checks that every Customer ID appears once, that the number of rows equals the number of customers with sales, and that no values are missing.

In [ ]:
# Each row should be one customer and nothing should be missing
print('Unique customer ids:', customers.index.is_unique)
print('Rows:', len(customers), '| Customers in sales:', sales['Customer ID'].nunique())
print('Missing values:', customers.isna().sum().sum())

### What I found

- Every Customer ID appears once, and the 5,853 rows match the 5,853 customers with sales
- There are no missing values, so each row is exactly one customer

## Step 17: Feature Dictionary

| Feature | Definition | Unit | What it means |
|---|---|---|---|
| `recency_days` | Days from the customer's last purchase to the snapshot date | days | How recently the customer bought. Smaller means more recent |
| `frequency` | Number of unique invoices (orders) | orders | How often the customer buys |
| `total_spend` | Sum of Quantity times Price over all sales lines | pounds | How much the customer spent in total |
| `avg_order_value` | total_spend divided by frequency | pounds per order | How big a typical order is |
| `avg_basket_units` | Total units bought divided by frequency | units per order | How many items go into a typical order |
| `unique_products` | Number of different stock codes bought | products | How varied the customer's buying is |
| `customer_age_days` | Days from the customer's first purchase to the snapshot date | days | How long the customer has been buying |
| `return_ratio` | Value of returned goods divided by total_spend | ratio | How much of their spending is returned |

## Step 18: Deciding How to Encode Categorical Columns

**Why:** The only categorical column left is `Country`, and I have to decide whether it goes into the clustering. One-hot encoding is a good choice when a nominal column has a few categories that really matter for similarity. Here it would add around 40 new columns, and about 90% of rows are from the United Kingdom, so those columns would be almost all zeros. Assigning numbers like UK = 1, Germany = 2 would be worse, because the algorithms would treat Germany as "twice" the UK and measure distances between countries that don't mean anything.

**Decision:** I'm not encoding `Country` for clustering. All eight features I built are numeric, so the groups will be based on buying behaviour only. I keep `Country` to describe the clusters afterwards.

## Step 19: Keeping Country for Describing the Clusters

**Why:** I want to compare the clusters by country later, but without letting country decide the groups. Each customer needs one country, so I'm using the country they bought from most often.

**What the code does:** Finds the most common country for each customer and saves it in a separate table called profile_vars, shows the five biggest, and counts how many customers bought from more than one country.

In [ ]:
# Most common country for each customer, kept only for describing clusters later
profile_vars = sales.groupby('Customer ID')['Country'].agg(lambda s: s.mode().iat[0]).to_frame('Main_country')
print(profile_vars['Main_country'].value_counts().head(5))

# How many customers bought from more than one country
print('Customers with more than one country:', (sales.groupby('Customer ID')['Country'].nunique() > 1).sum())

In [ ]:
# Saving both tables so the later steps can be rerun without rebuilding them
customers.to_csv('customers.csv')
profile_vars.to_csv('profile_vars.csv')